# CC3M：在 Colab A100 上从头训练 CLIP

先在「运行时 → 更改运行时类型」选择 A100。按顺序运行本 Notebook。

无需推送 GitHub：本地执行 `python scripts/export_colab.py`，下载生成的 `/tmp/clip-repr-colab.zip`，再上传到下面的代码单元。上传的压缩包包含当前未提交的代码修改。

数据要求：每个 tar 内，同一 key 的图片与 `.txt` 或包含 `caption`/`text` 字段的 `.json` 相邻存放，例如 `00001.jpg`、`00001.txt`。本 Notebook 不会重新下载 CC3M。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT = Path("/content/clip-repr")
if not (PROJECT / "train_cc3m.py").exists():
    from google.colab import files
    uploaded = files.upload()  # 选择 clip-repr-colab.zip
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("请选择一个项目 ZIP 文件")
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall("/content")
if not (PROJECT / "train_cc3m.py").exists():
    raise FileNotFoundError("ZIP 应包含 clip-repr/train_cc3m.py")
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(PROJECT / "requirements-colab.txt")], check=True)


In [ ]:
import torch
import torchvision
import shutil

print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
assert torch.cuda.is_available(), "请先切换到 GPU 运行时"
assert torch.cuda.is_bf16_supported(), "当前 GPU 不支持 BF16，请选择 A100"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Local free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))


## 设置路径与训练参数

修改 `DATA_ROOT` 为 Drive 上 tar 分片的实际目录。分片会递归查找。若已有独立验证目录，填写 `VAL_ROOT`，且训练目录不能包含验证目录；否则固定留出 `VAL_SHARDS` 个分片。

`BATCH_SIZE=128`、`NUM_WORKERS=2` 是短跑的起点，依据实测显存和速度调整。正式续训时保持 batch、总步数、warmup、验证样本数、seed、precision 和 shuffle buffer 一致。`TOTAL_STEPS` 是更新次数，不代表遍历数据的轮数。


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/cc3m")  # 修改这里
VAL_ROOT = None  # 例如 Path("/content/drive/MyDrive/cc3m_val")
VAL_SHARDS = 1
EXPERIMENT = "cc3m_a100_01"
LOCAL_OUTPUT = Path("/content/runs") / EXPERIMENT
DRIVE_OUTPUT = Path("/content/drive/MyDrive/clip-repr-runs") / EXPERIMENT
CACHE_DIR = Path("/content/cc3m-cache")

BATCH_SIZE = 128
NUM_WORKERS = 2
TOTAL_STEPS = 30000
WARMUP_STEPS = 1000
SAVE_EVERY = 500
EVAL_EVERY = 1000
VAL_SAMPLES = 1024
RESUME = False  # 新 Colab 会话恢复训练时改成 True


## 检查数据

先查看首个分片的图文对。若出现大量 INVALID，先确认 tar 内部格式，不要开始正式训练。读取过程中损坏图片和无效 caption 会被跳过并计数；tar 本身损坏或读取失败会直接报错。

本地暂存按当前分片进行，训练进程与验证各自使用临时目录。减少 workers 可降低同时缓存的空间，但每个完整 tar 仍必须能放入本地磁盘。正常退出会清理缓存；强制终止后可通过重新连接运行时清理。


In [ ]:
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"请修改 DATA_ROOT：{DATA_ROOT}")
shards = list(DATA_ROOT.rglob("*.tar"))
if VAL_ROOT is not None:
    shards += list(VAL_ROOT.rglob("*.tar"))
if not shards:
    raise FileNotFoundError("未找到 .tar 分片")
largest = max(path.stat().st_size for path in shards)
required = (max(1, NUM_WORKERS) + 1) * largest + 15 * 2**30
free = shutil.disk_usage("/content").free
print("Shards:", len(shards), "| largest GiB:", round(largest / 2**30, 2))
print("Estimated staging + checkpoint reserve GiB:", round(required / 2**30, 1))
if free < required:
    raise RuntimeError("本地空间不足，请减少 NUM_WORKERS 或改用更小分片")
subprocess.run([sys.executable, "-m", "data.cc3m", str(DATA_ROOT), "--limit", "10"],
               cwd=PROJECT, check=True)


In [ ]:
def training_command(save_dir, workers=NUM_WORKERS):
    command = [
        sys.executable, "-u", "train_cc3m.py",
        "--data-root", str(DATA_ROOT), "--save-dir", str(save_dir),
        "--cache-dir", str(CACHE_DIR),
        "--context-length", "77", "--precision", "bf16", "--device", "cuda",
        "--batch-size", str(BATCH_SIZE), "--num-workers", str(workers),
        "--total-steps", str(TOTAL_STEPS), "--warmup-steps", str(WARMUP_STEPS),
        "--save-every", str(SAVE_EVERY), "--eval-every", str(EVAL_EVERY),
        "--val-samples", str(VAL_SAMPLES), "--val-shards", str(VAL_SHARDS),
        "--shuffle-buffer", "512", "--seed", "42",
    ]
    if VAL_ROOT is not None:
        command += ["--val-root", str(VAL_ROOT)]
    return command


## 短跑验证

只跑 10 次更新，模型使用正式结构；检查 loss 是否有限、峰值显存和 samples/s。短跑 checkpoint 只保存到本地的独立目录。若调整 batch 后重试，使用新的短跑目录。


In [ ]:
from datetime import datetime
smoke_dir = Path("/content/runs") / ("smoke_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
command = training_command(smoke_dir)
command += ["--stop-after-steps", "10", "--log-every", "1"]
subprocess.run(command, cwd=PROJECT, check=True)


## 正式训练 / 恢复训练

首次运行保持 `RESUME=False`。恢复时设为 `True`，保持原来的实验名与训练参数。每隔 SAVE_EVERY 步保存 last.pt，验证后还会保存更优的 best.pt，并同步日志、配置和 TensorBoard 到 Drive。

恢复保留模型、优化器、学习率和全局步数；数据从下一次分片打乱开始，可能重复读取部分样本。Colab 强制断开时使用最近成功同步的 checkpoint。复制采用临时文件再重命名；这不能保证云端已在断电前完成持久化。

固定验证池使用所有候选计算双向 Recall@1/5/10，而非 batch 内检索。候选数量会记录在日志中。


In [ ]:
command = training_command(LOCAL_OUTPUT)
command += ["--drive-save-dir", str(DRIVE_OUTPUT), "--log-every", "50"]
if RESUME:
    checkpoint = DRIVE_OUTPUT / "last.pt"
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)
    command += ["--resume", str(checkpoint)]
subprocess.run(command, cwd=PROJECT, check=True)


## 查看曲线

训练日志会打印峰值显存、吞吐量、学习率、loss 与验证检索指标。TensorBoard 读取已同步到 Drive 的记录。


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT / "tensorboard"}


## 可选：CIFAR-10 零样本评估

加载 checkpoint 同目录的 config.py，自动采用 CC3M 的 CLIP 预处理。以下评估使用 CIFAR-10 测试集。


In [ ]:
subprocess.run([
    sys.executable, "zero_shot.py",
    "--checkpoint", str(DRIVE_OUTPUT / "best.pt"),
    "--data_root", "/content/cifar10",
    "--batch_size", "128", "--num_workers", "2",
], cwd=PROJECT, check=True)
